<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Inspects and lightly normalizes the course-offering source table for future candidate-course filtering.

**Notebook Shape:** 12 cells (11 code, 1 markdown).

**Inputs / Data Sources:**
- `df_raw=pd.read_parquet(RAW_DIR / "v_sch_course_offers.parquet")`

**Outputs / Side Effects:**
- `No explicit persisted output detected; side effects are limited to notebook display state unless cells are edited.`

**Logic Flow:**
1. Load raw course-offer parquet.
2. Run ID quality reports and normalization helpers.
3. Inspect offer keys and term coverage.

**Maintainability Notes:** Offering data likely belongs in candidate generation; keep term filtering rules explicit before using it in recommendations.

# V_SCH_COURSE_OFFER Cleaning

Scope: clean and prepare course-offer availability rows already loaded in `df_raw`. This notebook does not access databases, credentials, parquet files, or external data. It creates in-memory pandas DataFrames only.


In [ ]:
# Cell 1: copy raw DataFrame and standardize column names
import pandas as pd

from src.cleaning_utils import integer_like_report, normalize_id_columns
from src.paths import RAW_DIR

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

TABLE_NAME = "V_SCH_COURSE_OFFER"
df_raw = pd.read_parquet(RAW_DIR / "v_sch_course_offers.parquet")
assert "df_raw" in globals(), "df_raw must already be loaded before running this notebook."


def normalize_column_names(columns: pd.Index) -> pd.Index:
    normalized = (
        pd.Index(columns)
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"[^0-9a-z]+", "_", regex=True)
        .str.strip("_")
    )
    assert not normalized.duplicated().any(), "Column-name normalization created duplicate column names."
    return normalized


def build_null_report(frame: pd.DataFrame) -> pd.DataFrame:
    row_count = len(frame)
    null_count = frame.isna().sum()
    return (
        pd.DataFrame(
            {
                "column": frame.columns,
                "dtype": [str(dtype) for dtype in frame.dtypes],
                "null_count": [int(null_count[column]) for column in frame.columns],
                "null_percent": [
                    round(float(null_count[column] / row_count * 100), 4) if row_count else 0.0
                    for column in frame.columns
                ],
            }
        )
        .sort_values(["null_count", "column"], ascending=[False, True])
        .reset_index(drop=True)
    )


def clean_text_series(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip()
    empty_like = cleaned.eq("") | cleaned.str.lower().isin(["nan", "none", "null", "<na>"])
    return cleaned.mask(empty_like, pd.NA)


def to_nullable_int(series: pd.Series, column_name: str) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    non_numeric_mask = series.notna() & numeric.isna()
    fractional_mask = numeric.notna() & ((numeric % 1) != 0)

    assert not non_numeric_mask.any(), f"{column_name} contains non-numeric values."
    assert not fractional_mask.any(), f"{column_name} contains fractional/suffix values; cannot safely cast to Int64."

    return numeric.astype("Int64")


def build_id_issue_sample(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    samples = []
    for column in columns:
        numeric = pd.to_numeric(frame[column], errors="coerce")
        non_numeric_mask = frame[column].notna() & numeric.isna()
        fractional_mask = numeric.notna() & ((numeric % 1) != 0)
        issue_values = frame.loc[non_numeric_mask | fractional_mask, column].drop_duplicates().head(25)
        if len(issue_values):
            samples.append(
                pd.DataFrame(
                    {
                        "column": column,
                        "value": issue_values.astype("string").to_list(),
                        "issue": "non_integer_or_fractional_value",
                    }
                )
            )
    return pd.concat(samples, ignore_index=True) if samples else pd.DataFrame(columns=["column", "value", "issue"])


df = df_raw.copy()
raw_shape = df.shape
raw_row_count = len(df)
df.columns = normalize_column_names(df.columns)
df["_source_row_order"] = range(len(df))

required_input_columns = [
    "level_category_id",
    "part_id",
    "department_id",
    "faculty_id",
    "course_id",
    "course_type_id",
    "course_name_sl",
    "faculty_name_sl",
    "department_name_sl",
    "course_credits",
    "allow_register",
]

missing_input_columns = sorted(set(required_input_columns) - set(df.columns))
extra_input_columns = sorted(set(df.columns) - set(required_input_columns) - {"_source_row_order"})

print("Table:", TABLE_NAME)
print("Raw shape:", raw_shape)
print("Working shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing input columns:", missing_input_columns)
print("Extra input columns:", extra_input_columns)

assert not missing_input_columns, f"Missing required input columns: {missing_input_columns}"

In [ ]:
# Cell 2: basic reports before cleaning
null_report_before_cleaning = build_null_report(df[required_input_columns])
allow_register_raw_value_counts = df["allow_register"].value_counts(dropna=False)
sample_rows_before_cleaning = df[required_input_columns].head(10).copy()

print("DataFrame info before cleaning:")
df[required_input_columns].info()

print("\nNull report before cleaning:")
display(null_report_before_cleaning)

print("\nallow_register value counts before cleaning:")
display(allow_register_raw_value_counts)

print("\nSample rows before cleaning:")
display(sample_rows_before_cleaning)


In [ ]:
# Cell 3: clean strings
text_columns = [
    "course_name_sl",
    "faculty_name_sl",
    "department_name_sl",
    "allow_register",
]

for column in text_columns:
    df[column] = clean_text_series(df[column])

df["allow_register"] = df["allow_register"].str.upper()

string_cleaning_report = pd.DataFrame(
    {
        "column": text_columns,
        "dtype_after_cleaning": [str(df[column].dtype) for column in text_columns],
        "null_count_after_cleaning": [int(df[column].isna().sum()) for column in text_columns],
    }
)

print("String cleaning complete.")
display(string_cleaning_report)
print("allow_register value counts after string cleaning:")
display(df["allow_register"].value_counts(dropna=False))


In [ ]:
# Cell 4: validate allow_register
valid_allow_register_values = {"Y", "N"}

invalid_allow_register_mask = df["allow_register"].notna() & ~df["allow_register"].isin(valid_allow_register_values)
invalid_allow_register_report = df.loc[invalid_allow_register_mask].copy()

print("Invalid allow_register rows:", len(invalid_allow_register_report))
if len(invalid_allow_register_report):
    display(invalid_allow_register_report)
else:
    print("allow_register contains only Y, N, or missing values.")

assert invalid_allow_register_report.empty, "Unexpected allow_register values found. See invalid_allow_register_report."


In [ ]:
# Cell 5: create registration availability flag
# Y means available. N and missing values mean unavailable.
df["is_registration_allowed"] = df["allow_register"].eq("Y").fillna(False).astype("boolean")

is_registration_allowed_distribution = df["is_registration_allowed"].value_counts(dropna=False)
allow_register_flag_crosscheck = pd.crosstab(
    df["allow_register"].fillna("<NA>"),
    df["is_registration_allowed"],
    dropna=False,
)

print("is_registration_allowed distribution:")
display(is_registration_allowed_distribution)

print("allow_register vs is_registration_allowed:")
display(allow_register_flag_crosscheck)

assert df["is_registration_allowed"].notna().all(), "is_registration_allowed must not contain null values."
assert df.loc[df["allow_register"].eq("Y"), "is_registration_allowed"].all(), "Y rows must be allowed."
assert not df.loc[df["allow_register"].ne("Y") | df["allow_register"].isna(), "is_registration_allowed"].any(), "N/missing rows must be unavailable."


In [ ]:
# Cell 6: ID handling with project-approved helper functions
simple_integer_id_columns = [
    "level_category_id",
    "part_id",
    "course_type_id",
]

suffix_sensitive_id_columns = [
    "department_id",
    "faculty_id",
    "course_id",
]

all_id_columns = simple_integer_id_columns + suffix_sensitive_id_columns

id_validation_report = pd.DataFrame([integer_like_report(df, column) for column in all_id_columns])
id_issue_sample_report = build_id_issue_sample(df, all_id_columns)

print("ID validation report before conversion:")
display(id_validation_report)

print("ID non-integer/fractional value samples before conversion:")
display(id_issue_sample_report)

# These IDs are academic/category codes and are safe to cast only after validation rejects suffix/fractional values.
for column in simple_integer_id_columns:
    df[column] = to_nullable_int(df[column], column)

# These IDs can carry meaningful decimal suffixes such as .111, so preserve them as normalized strings.
df = normalize_id_columns(df, suffix_sensitive_id_columns)

id_dtype_report_after_cleaning = pd.DataFrame(
    {
        "column": all_id_columns,
        "dtype_after_cleaning": [str(df[column].dtype) for column in all_id_columns],
        "null_count_after_cleaning": [int(df[column].isna().sum()) for column in all_id_columns],
    }
)

print("ID dtypes after cleaning:")
display(id_dtype_report_after_cleaning)


In [ ]:
# Cell 7: numeric cleaning for course_credits
course_credits_numeric = pd.to_numeric(df["course_credits"], errors="coerce")
invalid_course_credits_mask = df["course_credits"].notna() & course_credits_numeric.isna()
invalid_course_credits_report = df.loc[invalid_course_credits_mask].copy()

# Fractional credits are valid and must be preserved.
fractional_course_credits_mask = course_credits_numeric.notna() & ((course_credits_numeric % 1) != 0)
fractional_course_credits_report = df.loc[fractional_course_credits_mask].copy()

df["course_credits"] = course_credits_numeric.astype("Float64")

course_credits_describe = df["course_credits"].describe()
course_credits_value_counts = df["course_credits"].value_counts(dropna=False).sort_index()

print("Invalid course_credits rows coerced to NaN:", len(invalid_course_credits_report))
if len(invalid_course_credits_report):
    display(invalid_course_credits_report)

print("course_credits describe:")
display(course_credits_describe)

print("course_credits value counts:")
display(course_credits_value_counts)

print("Fractional course_credits rows:", len(fractional_course_credits_report))
display(fractional_course_credits_report.head(50))


In [ ]:
# Cell 8: duplicate checks and duplicate removal
logical_key = [
    "part_id",
    "faculty_id",
    "department_id",
    "level_category_id",
    "course_id",
]

extended_key = [
    "part_id",
    "faculty_id",
    "department_id",
    "level_category_id",
    "course_id",
    "course_type_id",
]

duplicate_logical_key_report = (
    df.loc[df.duplicated(logical_key, keep=False)]
    .sort_values(logical_key + ["_source_row_order"])
    .copy()
)

duplicate_extended_key_report = (
    df.loc[df.duplicated(extended_key, keep=False)]
    .sort_values(extended_key + ["_source_row_order"])
    .copy()
)

exact_duplicate_compare_columns = [column for column in df.columns if column != "_source_row_order"]
exact_duplicate_report = df.loc[df.duplicated(exact_duplicate_compare_columns, keep=False)].copy()
exact_duplicate_rows_removed = int(df.duplicated(exact_duplicate_compare_columns, keep="first").sum())

print("Duplicate rows on logical key:", len(duplicate_logical_key_report))
print("Duplicate rows on extended key:", len(duplicate_extended_key_report))
print("Exact duplicate rows that will be removed:", exact_duplicate_rows_removed)

display(duplicate_logical_key_report)
display(duplicate_extended_key_report)

rows_before_duplicate_removal = len(df)
df = df.drop_duplicates(subset=exact_duplicate_compare_columns).copy()

# For remaining non-exact logical-key duplicates, keep highest registration priority: Y > N > missing.
df["_allow_priority"] = df["allow_register"].map({"Y": 2, "N": 1}).fillna(0).astype("int64")
rows_before_priority_dedup = len(df)

df = (
    df.sort_values(
        logical_key + ["_allow_priority", "_source_row_order"],
        ascending=[True] * len(logical_key) + [False, True],
    )
    .drop_duplicates(subset=logical_key, keep="first")
    .sort_values("_source_row_order")
    .drop(columns=["_allow_priority"])
    .reset_index(drop=True)
)

priority_duplicate_rows_removed = rows_before_priority_dedup - len(df)
duplicate_rows_removed = rows_before_duplicate_removal - len(df)

duplicate_removal_summary = pd.DataFrame(
    {
        "metric": [
            "rows_before_duplicate_removal",
            "exact_duplicate_rows_removed",
            "priority_duplicate_rows_removed",
            "total_duplicate_rows_removed",
            "rows_after_duplicate_removal",
        ],
        "value": [
            rows_before_duplicate_removal,
            exact_duplicate_rows_removed,
            priority_duplicate_rows_removed,
            duplicate_rows_removed,
            len(df),
        ],
    }
)

display(duplicate_removal_summary)


In [ ]:
# Cell 9: final validation
critical_non_allow_register_columns = [
    "level_category_id",
    "part_id",
    "department_id",
    "faculty_id",
    "course_id",
    "course_type_id",
    "course_name_sl",
    "faculty_name_sl",
    "department_name_sl",
    "course_credits",
]

final_business_columns = [
    "part_id",
    "faculty_id",
    "department_id",
    "level_category_id",
    "course_id",
    "course_type_id",
    "course_credits",
    "allow_register",
    "is_registration_allowed",
    "course_name_sl",
    "faculty_name_sl",
    "department_name_sl",
]

null_report_after_cleaning = build_null_report(df[final_business_columns])
bad_null_rows = df.loc[df[critical_non_allow_register_columns].isna().any(axis=1)].copy()
remaining_invalid_allow_register = df.loc[
    df["allow_register"].notna() & ~df["allow_register"].isin(valid_allow_register_values)
].copy()
missing_final_columns = sorted(set(final_business_columns) - set(df.columns))

print("Null report after cleaning:")
display(null_report_after_cleaning)

print("Bad null rows in critical non-allow_register columns:", len(bad_null_rows))
if len(bad_null_rows):
    display(bad_null_rows)

assert not missing_final_columns, f"Missing final columns: {missing_final_columns}"
assert bad_null_rows.empty, "Critical non-allow_register columns contain nulls. See bad_null_rows."
assert remaining_invalid_allow_register.empty, "Unexpected allow_register values remain."
assert not df.duplicated(logical_key).any(), "Duplicate logical keys remain after duplicate handling."
assert df["is_registration_allowed"].notna().all(), "is_registration_allowed contains nulls."
assert pd.api.types.is_numeric_dtype(df["course_credits"]), "course_credits must be numeric."

validation_summary = pd.DataFrame(
    {
        "metric": [
            "raw_rows",
            "clean_rows_before_final_column_selection",
            "raw_columns",
            "clean_columns_before_final_column_selection",
            "duplicate_rows_removed",
            "bad_null_rows",
            "invalid_allow_register_rows",
            "remaining_logical_key_duplicates",
        ],
        "value": [
            raw_row_count,
            len(df),
            raw_shape[1],
            df.shape[1],
            duplicate_rows_removed,
            len(bad_null_rows),
            len(remaining_invalid_allow_register),
            int(df.duplicated(logical_key).sum()),
        ],
    }
)

display(validation_summary)
print("Raw row count:", raw_row_count)
print("Clean row count:", len(df))
print("Rows removed due to duplicates:", duplicate_rows_removed)


In [ ]:
# Cell 10: final output DataFrames
# Keep required business columns plus any project-standard helper columns if they exist.
id_helper_output_columns = [
    column
    for column in df.columns
    if column.startswith(("department_id_", "faculty_id_", "course_id_"))
]

final_columns = final_business_columns + [
    column for column in id_helper_output_columns if column not in final_business_columns
]

df_clean_course_offer = (
    df[final_columns]
    .sort_values(logical_key)
    .reset_index(drop=True)
    .copy()
)

df_available_course_offer = df_clean_course_offer.loc[
    df_clean_course_offer["is_registration_allowed"]
].copy()

print("df_clean_course_offer shape:", df_clean_course_offer.shape)
print("df_available_course_offer shape:", df_available_course_offer.shape)
print("Final columns:", df_clean_course_offer.columns.tolist())

print("Available-course filter example:")
print('available_courses_for_part = df_clean_course_offer[(df_clean_course_offer["part_id"] == target_part_id) & (df_clean_course_offer["is_registration_allowed"])]')

display(df_clean_course_offer.head())
display(df_available_course_offer.head())
